In [ ]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.borsaitaliana.it"
URL = BASE_URL + "/borsa/obbligazioni/mot/btp/lista.html?lang=it"

soup = BeautifulSoup(requests.get(URL).content, "html5lib")
pages = [int(y.split('page=')[-1]) for y in [x["href"] for x in soup.select("li.m-pagination__item a")] if 'page=' in y]
pages = list(set([1] + pages))
print(pages)

In [ ]:
btp_urls = []
for p in pages:
    url = URL + '&page=%s' % p
    soup = BeautifulSoup(requests.get(url).content, "html5lib")
    for sc in soup.select("td a[title*='Accedi alla scheda strumento']"):
        btp_urls.append(sc['href'])
btp_urls = list(set(btp_urls))
print(len(btp_urls))
print(btp_urls)

In [ ]:
from datetime import datetime

TAX = 0.125

btps = []
for btpu in btp_urls:
    url = BASE_URL + btpu
    soup = BeautifulSoup(requests.get(url).content, "html5lib")
    scad = soup.find(string="Scadenza").find_parent('tr').select("span.t-text.-right")[0].text.replace('\n', '').strip()
    prz = soup.find(string="Prezzo di riferimento").find_parent('tr').select("span.t-text.-right")[0].text.replace('\n', '').strip()
    if scad and prz:
        isin = btpu.split('/')[-1].split('.')[0]
        scad = datetime.strptime(scad[:6]+'20'+scad[6:], "%d/%m/%Y")
        years_to_end = round((scad - datetime.now()).total_seconds() / 60 / 60 / 24 / 365, 2)
        prz = float(prz.replace(',','.'))
        ced = soup.find(string="Tasso Cedola Periodale").find_parent('tr').select("span.t-text.-right")[0].text.replace('\n', '').strip()
        ced = ced and float(ced.replace(',','.')) or 0
        pced = soup.find(string="Periodicità cedola").find_parent('tr').select("span.t-text.-right")[0].text.replace('\n', '').strip()
        if pced == "Semestrale":
            ced *= 2
        elif pced == "Trimestrale":
            ced *= 4
        ren = soup.find(string="Rendimento effettivo a scadenza netto").find_parent('tr').select("span.t-text.-right")[0].text.replace(',', '.').strip()
        ren = ren and float(ren.replace(',','.')) or 0
        c_ren = ((100 - prz - max(0, (100 - prz) * TAX)) / years_to_end + ced * (1 - TAX)) / prz * 100
        btp = (url, isin, scad.strftime("%d/%m/%Y"), f"{years_to_end:.2f}".replace('.',','), f"{prz:.3f}".replace('.',','), f"{ced:.3f}".replace('.',','), f"{ren:.3f}".replace('.',','), f"{c_ren:.3f}".replace('.',','))
        btps.append(btp)
        print(btp)

In [7]:

import csv

header = ["Link", "ISIN", "Scadenza", "Durata (anni)", "Prezzo", "Cedola Annua", "Rendimento netto", "Rendimento netto calcolato"]

with open(f'btp_extract_{datetime.now():%Y%m%d%H%M}.csv', 'w', encoding='UTF8', newline='') as f:
    writer = csv.writer(f)

    # write the header
    writer.writerow(header)

    # write multiple rows
    writer.writerows(btps)